# ShadowWork-SHIFT | Final Hugging Face publication from Kaggle

**Purpose:** Publish SW12 v3 + SW17 v4 LoRA adapters, the **actual SW12/SW17 training datasets**, and SW12–SW17 evidence into the already-created public Hugging Face repositories, then add the two adapter repos to the existing required **model collection**. This notebook **does not train or evaluate** a model and does not change scientific results.

**Inputs:** Attach the existing Kaggle dataset `shaurya4667/shadowwork-12-closed-loop` (which contains `learner_corrective_v3_adapter/` and `rehearsal_v3_adapter/`, plus training rows/traces). Optionally attach the original SW13, SW14, SW15 and SW16 result ZIPs as a Kaggle input dataset, and/or the curated 5MB `ShadowWork_SHIFT_Publication_Evidence.zip` produced from Project source files. If not attached, SW12 models/data will still publish, but the extended evidence upload will be incomplete. All source checkpoint weights already exist; do **not** rerun experiments.

**Account/secret:** Set a Kaggle Secret named `HF_TOKEN` with a write-enabled Hugging Face access token for `i-am-shaurya05`; enable Internet in Kaggle. Never paste the token into code, notebooks, GitHub or logs. The public Plain SFT base and SW09 adapters have already been verified on HF, so this notebook avoids uploading them again.

**Limit:** Original model and dataset collections already exist. The optional collection-add operations are safe to repeat using `exists_ok=True`.

**SW17 input:** When running in the completed SW17 Kaggle session, use its saved `/kaggle/working/shadowwork/17_training_attribution` outputs; when running in a separate publisher session, attach the **saved SW17 output dataset**. The publisher will locate it using `learner_corrective_v4_training_metadata.json`. Both SW17 adapters and the finalized scorecard/paired tests must be present. **No training will be run.**

**Important:** The SW17 evidence ZIP does **not contain the trained v4 adapter weights** (the SW17 export excludes `*_adapter/` directories). Keep the original Kaggle runtime's v4 adapter folders, or attach them separately as a Kaggle dataset. The notebook verifies this before any upload. A standalone evidence ZIP by itself is insufficient for full v4 model publication.


In [ ]:
%pip install -q "huggingface_hub>=0.34,<2"

## 1. Editable settings and authentication

In [ ]:
from pathlib import Path
import json
import hashlib
import shutil
import zipfile
from huggingface_hub import HfApi, login, hf_hub_download
from kaggle_secrets import UserSecretsClient

OWNER = "i-am-shaurya05"
BASE_MODEL_ID = f"{OWNER}/shadowwork-shift-plain-sft"
CORRECTIVE_MODEL_ID = f"{OWNER}/shadowwork-shift-corrective-v3"
REHEARSAL_MODEL_ID = f"{OWNER}/shadowwork-shift-rehearsal-v3"
CORRECTIVE_V4_ID = f"{OWNER}/shadowwork-shift-corrective-v4"
REHEARSAL_V4_ID = f"{OWNER}/shadowwork-shift-rehearsal-v4"
DATASET_REPO_ID = f"{OWNER}/shadowwork-shift-datasets"

MODEL_COLLECTION_SLUG = f"{OWNER}/shadowwork-shift-models-and-frozen-adapters-6af5c8a5a548fb703ab420ac"
DATA_COLLECTION_SLUG = f"{OWNER}/shadowwork-shift-training-data-and-audit-evidence-6af5c8a51d44c02ee6e19812"

SW12_ROOT = Path("/kaggle/input/datasets/shaurya4667/shadowwork-12-closed-loop")
# Optional directory containing previous result ZIP archives (or nested below it).
RESULTS_ROOT = Path("/kaggle/input")
SW17_ROOT = Path("/kaggle/working/shadowwork/17_training_attribution")

# The SW17 evidence ZIP deliberately excludes trained adapter directories.
# It contains row hashes, precommitted tasks, rollouts, and evaluation traces.
# For full publication, the two trained v4 adapter folders must also exist,
# either in the original Kaggle working directory or as separate Kaggle inputs.
if not (SW17_ROOT / "precommit.json").is_file():
    candidates = []
    for file in RESULTS_ROOT.rglob("precommit.json"):
        try:
            check = json.loads(file.read_text())
        except (OSError, ValueError):
            continue
        if check.get("experiment") == "SW17-learner-repair-vs-matched-expert":
            candidates.append(file.parent)

    if not candidates:
        evidence_zips = sorted(RESULTS_ROOT.rglob("*17*Attribution*Evidence*.zip"))
        assert len(evidence_zips) == 1, (
            "Attach the exact SW17 evidence ZIP as a Kaggle input, or "
            "publish in the original Kaggle session.", evidence_zips
        )
        extract_to = STAGING_ROOT / "extracted_sw17_evidence"
        if extract_to.exists():
            shutil.rmtree(extract_to)
        extract_to.mkdir(parents=True)
        with zipfile.ZipFile(evidence_zips[0]) as archive:
            assert archive.testzip() is None, "SW17 evidence ZIP failed CRC validation"
            for item in archive.infolist():
                assert not Path(item.filename).is_absolute()
                assert ".." not in Path(item.filename).parts
            archive.extractall(extract_to)
        candidates = []
        for file in extract_to.rglob("precommit.json"):
            check = json.loads(file.read_text())
            if check.get("experiment") == "SW17-learner-repair-vs-matched-expert":
                candidates.append(file.parent)
    assert len(candidates) == 1, (
        "Could not locate one original SW17 experiment directory", candidates
    )
    SW17_ROOT = candidates[0]

assert (SW17_ROOT / "policy_summary.csv").is_file(), (
    "SW17 scoring and paired tests must be complete before publication."
)
assert (SW17_ROOT / "paired_tests.csv").is_file()
assert (SW17_ROOT / "final_provenance.json").is_file()
assert (SW17_ROOT / "learning_gate.json").is_file()

SW17_ADAPTERS = {}
for arm in ("learner_corrective_v4", "rehearsal_v4"):
    name = arm + "_adapter"
    candidate = SW17_ROOT / name
    if not (candidate / "adapter_model.safetensors").is_file():
        found = [
            file.parent for file in RESULTS_ROOT.rglob("adapter_model.safetensors")
            if file.parent.name == name
        ]
        assert len(found) == 1, (
            f"Missing trained {name} LoRA weights. The SW17 evidence ZIP "
            "EXCLUDES model adapters. Keep the original Kaggle working files "
            "or attach both v4 adapter directories as an input dataset.", found
        )
        candidate = found[0]
    assert (candidate / "adapter_config.json").is_file(), candidate
    SW17_ADAPTERS[arm] = candidate

print("Using exact SW17 evidence:", SW17_ROOT)
print("Using existing v4 adapters:", SW17_ADAPTERS)
STAGING_ROOT = Path("/kaggle/working/shift_publication")
STAGING_ROOT.mkdir(parents=True, exist_ok=True)

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
assert HF_TOKEN, "Add a write-enabled HF_TOKEN in Kaggle Secrets"
api = HfApi(token=HF_TOKEN)
identity = api.whoami()
print("Publishing as:", identity.get("name"))
assert identity.get("name") == OWNER, "HF_TOKEN must belong to the expected account"
assert SW12_ROOT.is_dir(), f"Attach SW12 source dataset: {SW12_ROOT}"

# Verify existing public exact base checkpoint, not a similar upstream model.
base_info = api.model_info(BASE_MODEL_ID, files_metadata=False)
assert not base_info.private and not base_info.gated, "Base checkpoint must be public and ungated"
print("Verified existing public base:", BASE_MODEL_ID)


## 2. Verify original adapter files, publish exact weights with corrected public base linkage

In [ ]:
ADAPTERS = {
    "learner_corrective_v3": (CORRECTIVE_MODEL_ID, "learner_corrective_v3_adapter"),
    "rehearsal_v3": (REHEARSAL_MODEL_ID, "rehearsal_v3_adapter"),
}

for arm, (repo_id, relative_folder) in ADAPTERS.items():
    source_adapter = SW12_ROOT / relative_folder
    assert source_adapter.is_dir(), f"Missing {source_adapter}"
    assert (source_adapter / "adapter_model.safetensors").is_file()
    assert (source_adapter / "adapter_config.json").is_file()
    original_config = json.loads((source_adapter / "adapter_config.json").read_text())
    original_base = original_config.get("base_model_name_or_path", "")
    assert "shadowork_123" in original_base, f"Unexpected base in {arm}: {original_base}"

    model_dir = STAGING_ROOT / "model_releases" / arm
    model_dir.mkdir(parents=True, exist_ok=True)
    for name in ("adapter_model.safetensors", "adapter_config.json", "tokenizer.json", "tokenizer_config.json", "chat_template.jinja"):
        file = source_adapter / name
        if file.is_file():
            shutil.copy2(file, model_dir / name)

    original_sha = hashlib.sha256((source_adapter / "adapter_model.safetensors").read_bytes()).hexdigest()
    new_config = dict(original_config)
    new_config["base_model_name_or_path"] = BASE_MODEL_ID
    (model_dir / "adapter_config.json").write_text(json.dumps(new_config, indent=2) + "\n")
    (model_dir / "provenance").mkdir(exist_ok=True)
    (model_dir / "provenance" / "original_adapter_config.json").write_text(json.dumps(original_config, indent=2) + "\n")
    (model_dir / "provenance" / "original_weights_sha256.txt").write_text(original_sha + "\n")

    description = "learner-state corrective training" if arm == "learner_corrective_v3" else "matched ordinary rehearsal training"
    card = f"""---
library_name: peft
pipeline_tag: text-generation
license: cc-by-nc-4.0
base_model: {BASE_MODEL_ID}
base_model_relation: adapter
tags:
- lexsi
- agenttune
- auditkit
- shadowwork-shift
- peft
- lora
---

# ShadowWork-SHIFT: {arm}

This PEFT LoRA adapter is the **SW12 v3 continuation** trained by {description} on synthetic stateful tool-agent workflow tasks. Both v3 arms start from the frozen SW09 corrective adapter and share equal optimization budgets and matched training rows. The exact, non-commercial custom base is [{BASE_MODEL_ID}](https://huggingface.co/{BASE_MODEL_ID}). It cannot be replaced with the upstream Cohere base without changing the scientific comparison.

**Loading:** Use Transformers with the linked custom full checkpoint and PEFT's `PeftModel.from_pretrained(base_model, "{repo_id}")`. The original experiment also needs the task simulator, JSON action format and deterministic controller to reproduce the reported workflow outcomes.

**Training:** 800 rows per arm, 80 optimization steps, LR 5e-5, global batch 8. Source training rows and traces are in the paired [dataset release](https://huggingface.co/datasets/{DATASET_REPO_ID}). Weights are unchanged from the original export. The Kaggle-specific adapter base path was corrected in metadata; its original value and original weights hash are preserved under `provenance/`.

**Evaluation, constant unified_v4 controller over 160 SW12 core/composed task instances:** corrective v3 154/160 strict; rehearsal v3 160/160 strict. With final budgeted_v5 recovery controller: corrective v3 156/160; rehearsal v3 160/160. These are controlled synthetic results; corrective v3 does **not** establish superiority over matched rehearsal or real-world generalization. Original evaluation code: https://github.com/githubshaurya/ShadowWork-SHIFT

**License:** CC BY-NC 4.0 inherited from Cohere Tiny Aya model terms; observe the upstream acceptable use policy. Non-commercial research use only under these terms.
"""
    (model_dir / "README.md").write_text(card)

    api.create_repo(repo_id, repo_type="model", private=False, exist_ok=True)
    print("Uploading", arm, "original weight SHA256", original_sha)
    api.upload_folder(folder_path=str(model_dir), repo_id=repo_id, repo_type="model", commit_message=f"Publish original {arm} adapter and provenance")
    public_info = api.model_info(repo_id)
    public_names = {item.rfilename for item in public_info.siblings}
    assert not public_info.private and not public_info.gated
    assert "adapter_model.safetensors" in public_names and "adapter_config.json" in public_names
    remote_config_path = hf_hub_download(repo_id, "adapter_config.json", repo_type="model", token=HF_TOKEN, force_download=True)
    assert json.loads(Path(remote_config_path).read_text())["base_model_name_or_path"] == BASE_MODEL_ID
    print("Published and verified:", f"https://huggingface.co/{repo_id}")


# SW17 v4 model adapters: publish exactly the already-trained LoRA tensors.
V4_ADAPTERS = {
    "learner_corrective_v4": (CORRECTIVE_V4_ID, "learner_corrective_v4_adapter"),
    "rehearsal_v4": (REHEARSAL_V4_ID, "rehearsal_v4_adapter"),
}
for arm, (repo_id, folder_name) in V4_ADAPTERS.items():
    original = SW17_ADAPTERS[arm]
    assert (original / "adapter_model.safetensors").is_file(), original
    assert (original / "adapter_config.json").is_file(), original
    training = json.loads((SW17_ROOT / f"{arm}_training_metadata.json").read_text())
    original_rows = (SW17_ROOT / f"{arm}_training.jsonl").read_bytes()
    assert hashlib.sha256(original_rows).hexdigest() == training["training_sha256"]
    assert training["arm"] == arm and training["steps"] == 80
    assert training["train_rows"] == 800 and training["global_batch"] == 8
    config = json.loads((original / "adapter_config.json").read_text())
    assert config.get("peft_type", "").upper() == "LORA", "Unexpected PEFT adapter type"

    model_dir = STAGING_ROOT / "model_releases" / arm
    if model_dir.exists():
        shutil.rmtree(model_dir)
    shutil.copytree(original, model_dir)
    original_sha = hashlib.sha256((original / "adapter_model.safetensors").read_bytes()).hexdigest()
    (model_dir / "provenance").mkdir(exist_ok=True)
    (model_dir / "provenance" / "original_adapter_config.json").write_text(
        json.dumps(config, indent=2) + "\n")
    (model_dir / "provenance" / "original_weights_sha256.txt").write_text(original_sha + "\n")
    config["base_model_name_or_path"] = BASE_MODEL_ID
    (model_dir / "adapter_config.json").write_text(json.dumps(config, indent=2) + "\n")
    description = "learner-visited corrective" if arm == "learner_corrective_v4" else "matched ordinary rehearsal"
    model_card = f"""---
library_name: peft
pipeline_tag: text-generation
base_model: {BASE_MODEL_ID}
base_model_relation: adapter
license: cc-by-nc-4.0
tags:
- lexsi
- shadowwork-shift
- lora
---
# ShadowWork-SHIFT {arm}

An **already trained** LoRA adapter from SW17 {description} continuation. Same frozen original SW09 adapter start, 800 training rows, 400 shared expert anchors, 80 optimization steps, identical LR 5e-5 and global batch 8. This release does NOT retrain weights. Original adapter configuration and unchanged weight SHA-256 are included in provenance.

Under identical four scripted initial read actions and **no further repair controller**, SW17 strict success on the 80-case synthetic shifted-workflow holdout was: frozen SW09 **31/80**, corrective v4 **65/80**, rehearsal v4 **65/80**, previous rehearsal v3 **57/80**. Corrective and rehearsal each improved 34 tasks over the frozen model, but the correction-specific curriculum showed **no advantage** over matched rehearsal. Critically, both v4 models made **8 unsafe proposals each** on forbidden-injection cases; the runtime tool gate blocked them. No external-environment generalization is established.

Training rows, generation process, split hashes, model failure traces, and scorecards: [dataset](https://huggingface.co/datasets/{DATASET_REPO_ID}), [methodology](https://github.com/githubshaurya/ShadowWork-SHIFT/blob/main/docs/DATASET_METHOD.md).

Exact custom base: https://huggingface.co/{BASE_MODEL_ID}. Upstream generic checkpoints are not equivalent.
"""
    (model_dir / "README.md").write_text(model_card)
    api.create_repo(repo_id, repo_type="model", private=False, exist_ok=True)
    api.upload_folder(folder_path=str(model_dir), repo_id=repo_id, repo_type="model",
                      commit_message=f"Publish unchanged SW17 {arm} trained LoRA adapter")
    remote = api.model_info(repo_id, files_metadata=True)
    assert not remote.private and not remote.gated
    siblings = {item.rfilename: item for item in remote.siblings}
    assert "adapter_model.safetensors" in siblings and "adapter_config.json" in siblings
    if getattr(siblings["adapter_model.safetensors"], "lfs", None):
        assert siblings["adapter_model.safetensors"].lfs.sha256 == original_sha
    config_remote = hf_hub_download(repo_id, "adapter_config.json", token=HF_TOKEN, force_download=True)
    assert json.loads(Path(config_remote).read_text())["base_model_name_or_path"] == BASE_MODEL_ID
    print("V4 model published:", repo_id, "SHA256:", original_sha, flush=True)


## 3. Publish SW12 training metadata, datasets, exact scored episodes, and optional SW13–SW16 evidence

In [ ]:
api.create_repo(DATASET_REPO_ID, repo_type="dataset", private=False, exist_ok=True)

records = STAGING_ROOT / "dataset_release"
if records.exists():
    shutil.rmtree(records)
records.mkdir(parents=True, exist_ok=True)

# Exact original SW12 rows and scored evaluation JSONs are already mounted as a Kaggle dataset.
# Exclude very large duplicate tokenized Arrow data and all model adapters.
source_files = [
    "all_episode_scorecard.csv", "final_provenance.json", "grouped_summary.csv",
    "precommit.json", "selection_audit.json", "train_tasks.jsonl",
    "dev_tasks.jsonl", "sealed_tasks.jsonl",
    "learner_corrective_v3_training.jsonl", "rehearsal_v3_training.jsonl",
    "learner_corrective_v3_training_metadata.json", "rehearsal_v3_training_metadata.json",
    "learner_corrective_v3_tokenized_source_sha256.txt", "rehearsal_v3_tokenized_source_sha256.txt",
]
for name in source_files:
    file = SW12_ROOT / name
    assert file.is_file(), f"Expected SW12 evidence file: {file}"
    dest = records / "sw12" / name
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(file, dest)

sw12_episodes = sorted((SW12_ROOT / "evaluation").rglob("*.json"))
assert len(sw12_episodes) >= 200, "Could not locate expected SW12 evaluated episodes"
for file in sw12_episodes:
    dest = records / "sw12" / "evaluation" / file.relative_to(SW12_ROOT / "evaluation")
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(file, dest)
print("SW12 evaluated episodes:", len(sw12_episodes))

# The learner-visited rollouts are indispensable for auditing how corrections
# were selected. Publish them rather than only the final 800-row JSONL files.
sw12_collection = sorted((SW12_ROOT / "collection").glob("*.json"))
assert len(sw12_collection) == 160, "Expected 160 frozen-learner training rollouts in SW12"
for file in sw12_collection:
    dest = records / "sw12" / "collection" / file.name
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(file, dest)
print("SW12 train-only learner rollouts included:", len(sw12_collection))

# Verify the original SW12 split commitment and per-arm source hashes.
sw12_precommit = json.loads((SW12_ROOT / "precommit.json").read_text())
for split in ("train", "dev", "sealed"):
    source_bytes = (SW12_ROOT / f"{split}_tasks.jsonl").read_bytes()
    assert hashlib.sha256(source_bytes).hexdigest() == sw12_precommit["split_sha256"][split]
sw12_selection = json.loads((SW12_ROOT / "selection_audit.json").read_text())
assert sw12_selection["selected_learner_states"] == 400
assert sw12_selection["selected_actual_corrections"] == 175
assert sw12_selection["rollouts"] == 160
for arm in ("learner_corrective_v3", "rehearsal_v3"):
    data = (SW12_ROOT / f"{arm}_training.jsonl").read_bytes()
    assert len(data.splitlines()) == 800
    assert hashlib.sha256(data).hexdigest() == sw12_selection[
        "corrective_sha256" if arm == "learner_corrective_v3" else "rehearsal_sha256"
    ]

# Publish exact SW17 saved files, not reconstructed/favorable labels.
sw17_precommit = json.loads((SW17_ROOT / "precommit.json").read_text())
assert sw17_precommit["experiment"] == "SW17-learner-repair-vs-matched-expert"
for split in ("train", "dev", "sealed"):
    data = (SW17_ROOT / f"{split}_tasks.jsonl").read_bytes()
    assert hashlib.sha256(data).hexdigest() == sw17_precommit["split_sha256"][split]
for arm in ("learner_corrective_v4", "rehearsal_v4"):
    training_data = (SW17_ROOT / f"{arm}_training.jsonl").read_bytes()
    meta = json.loads((SW17_ROOT / f"{arm}_training_metadata.json").read_text())
    assert len(training_data.splitlines()) == 800
    assert hashlib.sha256(training_data).hexdigest() == meta["training_sha256"]
sw17_source_files = [
    "train_tasks.jsonl", "dev_tasks.jsonl", "sealed_tasks.jsonl", "precommit.json",
    "selection_audit.json", "learner_corrective_v4_training.jsonl",
    "rehearsal_v4_training.jsonl", "learner_corrective_v4_training_metadata.json",
    "rehearsal_v4_training_metadata.json", "episode_scorecard.csv",
    "policy_summary.csv", "group_summary.csv", "unseen_schema_summary.csv",
    "paired_tests.csv", "learning_gate.json", "final_provenance.json",
]
for name in sw17_source_files:
    source = SW17_ROOT / name
    assert source.is_file(), f"Finish SW17 scoring/evidence export before publication: {source}"
    destination = records / "sw17" / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)

sw17_collection = sorted((SW17_ROOT / "collection").glob("*.json"))
assert len(sw17_collection) == 160, "Missing SW17 train-only learner rollouts"
for source in sw17_collection:
    dest = records / "sw17" / "collection" / source.name
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, dest)
sw17_episodes = sorted((SW17_ROOT / "evaluation" / "sealed").rglob("*.json"))
assert len(sw17_episodes) >= 240, (
    "Need frozen, corrective, and rehearsal scored traces, 80 each", len(sw17_episodes)
)
for source in sw17_episodes:
    dest = records / "sw17" / "evaluation" / "sealed" / source.relative_to(
        SW17_ROOT / "evaluation" / "sealed")
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, dest)
print("SW17 publishable training/evaluation records:", len(sw17_collection),
      "train-only rollouts;", len(sw17_episodes), "sealed episodes", flush=True)


# Option A: Attach our curated 4.9-MB evidence ZIP as a Kaggle dataset.
# Option B: Mount individual original SW13–SW16 result ZIPs; the same release will expose their original archives.
curated_archives = sorted(RESULTS_ROOT.rglob("ShadowWork_SHIFT_Publication_Evidence.zip"))
if curated_archives:
    with zipfile.ZipFile(curated_archives[0]) as source_zip:
        assert source_zip.testzip() is None
        source_zip.extractall(records / "archived_evidence")
    print("Included curated complete SW09/SW13–SW16 evidence:", curated_archives[0])
else:
    names = (
        "ShadowWork_13_Live_Results.zip",
        "ShadowWork_14_Controller_Repair_Results1.zip",
        "ShadowWork_15_Cross_Protocol_Recovery_Results.zip",
        "ShadowWork_16_Unified_Recovery_Results.zip",
        "ShadowWork_16_Budget_Regression_Results.zip",
    )
    missing = []
    for name in names:
        matches = sorted(RESULTS_ROOT.rglob(name))
        if not matches:
            missing.append(name)
            continue
        dest = records / "archived_evidence" / "source_archives" / name
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(matches[0], dest)
    if missing:
        print("WARNING: optional evidence archives absent:", missing)

# This is the full, audited dataset card, not an abbreviated marketing blurb.
readme = "---\npretty_name: \"ShadowWork-SHIFT: Stateful Tool-Agent Recovery Training and Evidence\"\nlanguage:\n- en\ntags:\n- agents\n- synthetic\n- tool-use\n- imitation-learning\n- corrective-training\n- auditkit\n- lexsi\nlicense: cc-by-nc-4.0\n---\n\n# ShadowWork-SHIFT: training datasets and evaluation evidence\n\n**What this is:** Synthetic, executable, stateful configuration-maintenance tasks for measuring failure recovery in tool-using language models. The release includes paired corrective and expert-rehearsal supervised datasets, original train/dev/sealed task splits, learner-visited traces, selection audits, and held-out evaluation evidence. It is not a production-log or human-annotated dataset.\n\n**Source code and methodology:** [GitHub](https://github.com/githubshaurya/ShadowWork-SHIFT) · [Detailed creation process](https://github.com/githubshaurya/ShadowWork-SHIFT/blob/main/docs/DATASET_METHOD.md).\n\n## Construction\n\n1. Use a seeded generator to create **eight SW12 failure families**, including no-op writes, conflicting commits, stale read-backs, false-positive acknowledgement and forbidden/injected requests. Vary one-check and two-check post-mutation evidence rules.\n2. **Precommit train/development/sealed task identities and SHA-256 hashes before evaluation**; keep all supervised examples on the training split.\n3. Execute the **frozen SW09 tool policy** on 160 train-only tasks and save real model action/observation histories.\n4. Relabel learner-visited observation prefixes with a **scripted, observation-driven reference policy** under the SQLite tool gate. Validate action and persisted-state evidence. This is a **single-round, DAgger-inspired** procedure, *not* a human oracle, full iterative DAgger, or model-based reward labeling.\n5. Select 400 learner-state rows stratified by failure group, verification protocol and reference action, prioritizing disagreements. Combine with 400 shared expert anchors. Build a **matched 800-row rehearsal control** using reference-policy states, with the same starting checkpoint and compute budget.\n6. Publish the untouched split files, exact prompts/completions, evidence traces, audits and SHA manifests so the comparison can be inspected.\n\n**SW12 audited statistics:** 192 train + 32 dev + 64 sealed tasks; 160 train-only model rollouts; 1,276 learner states; 387 disagreement candidates; 400 selected learner states, of which **175** differ from the frozen model's action; 800 unique rows per arm; **452** identical prompt/completion pairs across the two realized arms (400 planned shared anchors plus 52 additional matches). Those numbers are from `ShadowWork_12_Closed_Loop_Evidence.zip`, not targets invented for this card.\n\n**SW17:** When the corresponding trained-run archive is included, see `sw17/` for its separate **10-family, 240/40/80-task** distribution-shift evaluation and 800-row v4 training arms. SW17 is a follow-up within the same synthetic simulator family, *not* an external benchmark. Do not mix SW12 and SW17 held-out denominators.\n\n## Files\n\n| Directory | Purpose |\n|---|---|\n| `sw12/*_training.jsonl` | Exact two 800-row supervised training files (`prompt`, `completion`) |\n| `sw12/{train,dev,sealed}_tasks.jsonl` | Precommitted synthetic task splits |\n| `sw12/collection/` | Train-only frozen-learner rollouts |\n| `sw12/evaluation/` | Held-out action traces, tool observations and outcomes |\n| `sw12/selection_audit.json`, `precommit.json` | Label-selection details and split identities |\n| `sw12/*_training_metadata.json` | Model-init provenance and optimization settings |\n| `sw17/` | Optional finalized v4 training/evaluation records when uploaded |\n| `sha256_release_manifest.json` | Released-file checksums |\n\nThe original SW09 training and 70-task holdout evidence are published separately at [SW09 dataset](https://huggingface.co/datasets/i-am-shaurya05/shadowwork-shift-evidence-and-training).\n\n## Integrity and limitations\n\nThe SW12 training rows and split JSONLs are published byte-for-byte from the archived run. Relevant original SHA values appear in [the methodological audit](https://github.com/githubshaurya/ShadowWork-SHIFT/blob/main/docs/DATASET_METHOD.md). Labels are scripted and verified within this simulator, not independently human-vetted. Because tasks share generator rules, generalization to unseen real services is **not established**. Rehearsal v3 scored **47/64**, versus **46/64** for corrective v3 under the same SW12 model-centric evaluation: failure-specific examples did not demonstrate unique superiority.\n\n**Intended usage:** Study learned self-healing and matched controls for stateful tool agents, test training/holdout leakage, and reproduce recorded simulator outcomes. Not intended as a certification of safe real-world autonomous writes.\n\n**License:** CC BY-NC 4.0 for this curated research release, consistent with the noncommercial model family. Check the separately published checkpoint's own terms before using weights.\n\n## Related references\n\n- [Ross et al. (2011): Dataset Aggregation (DAgger)](https://proceedings.mlr.press/v15/ross11a.html), methodological inspiration only.\n- [Gebru et al.: Datasheets for Datasets](https://arxiv.org/abs/1803.09010), documentation framework.\n- [Hugging Face dataset cards](https://huggingface.co/docs/hub/en/datasets-cards), distribution documentation.\n\n**Maintainer:** [ShadowWork-SHIFT GitHub](https://github.com/githubshaurya/ShadowWork-SHIFT). Dataset and model artifacts are published independently from the codebase.\n"
(records / "README.md").write_text(readme)
(records / "DATASET_METHOD.md").write_text("# ShadowWork-SHIFT: how the training data were actually constructed\n\n**Status:** This is a description of the executed SW12 pipeline and the source-defined SW17 continuation. SW12 counts below were independently checked against the original `ShadowWork_12_Closed_Loop_Evidence.zip`. SW17's final dataset counts/holdout scores must be taken from its saved artifacts, not inferred from its intended configuration.\n\n## Purpose and provenance\n\nThe data are **synthetic stateful tool-agent workflows** designed to test whether a tool-using language model can recover from misleading acknowledgements, no-op writes, conflicts, stale reads, insufficient verification, unauthorized requests and injected instructions. The data were **not collected from production services, scraped from real user logs, or manually annotated by human operators**. Scenario specifications, fault injections, and a deterministic reference policy are authored for this controlled evaluation.\n\nThe pipeline resembles **one round of learner-state expert relabeling**, related in motivation to dataset aggregation (Ross, Gordon and Bagnell, [AISTATS 2011](https://proceedings.mlr.press/v15/ross11a.html)). It is **not full iterative DAgger**, and the scripted reference policy is not an independently validated optimal expert.\n\n## Executed SW12 data pipeline\n\n1. **Generate deterministic scenarios and holdouts.** Seed `20261113`; 8 failure families: normal, double conflict, double no-op, wrong-then-stale, two stale checks, false positive, injected forbidden, injected authorized. Each family contains both one-check and two-check verification protocols. Files: 192 training tasks, 32 development tasks, and 64 sealed evaluation tasks. The task JSONLs and their SHA-256 digests were written to `precommit.json` before learner collection and evaluation. Protocol action budget: 22.\n2. **Run the frozen learner on training tasks.** Start from the archived SW09 corrective adapter on the exact 7-shard custom Plain SFT base (not a substitute generic upstream checkpoint). Collect 160 real policy rollouts on training tasks, distributed across all 8 failure families; store model actions, executable tool observations, final scores and source-model provenance.\n3. **Generate reference actions using observable history.** A deterministic reference policy reads the ticket, runbook, health, approval, current mutation receipts and post-write checks. It selects an authorized next JSON tool action, with explicitly typed field/value checks and evidence requirements. It does not read a hidden simulator-success label to choose the next action. The implementation *does* know the benchmark protocol's check-count setting, communicated to the agent in the task instructions. The reference is executed on the same gated SQLite simulator to check feasibility.\n4. **Construct learner-state training examples.** Replay saved learner action histories against the environment, and for each eligible learner-visited observation prefix create an `(observed prompt, reference next action)` label. Compare the original learner proposal against that reference. Token-length eligibility uses the same chat template as training; labels are not sourced from sealed cases.\n5. **Build matched training arms.** Both arms have exactly **800 prompt-completion rows**, with **400 deliberately shared expert anchor rows** and 400 differentiated rows. Corrective arm: selected learner-visited prefixes, stratified by fault group, verification protocol and teacher action, prioritizing action disagreements. Rehearsal arm: ordinary reference-policy/expert states from training tasks. Both start from the **same frozen SW09 adapter**, with 80 training steps, LR `5e-5` and global batch 8. This is a *matched-size/compute* comparison, not a guarantee of identical example difficulty or token budgets.\n6. **Independently check heldout performance.** Use the same workflow, tool grammar and evaluation controller per model. Strict success checks persisted SQLite state, authorization and sufficient post-mutation evidence, not just whether a language model says “completed.” Keep original JSON episodes and scorecards. Controller interventions are separate from fine-tuning effects.\n\n## Independently audited SW12 facts\n\n| Measurement | Verified count |\n|---|---:|\n| Train / development / sealed task specifications | 192 / 32 / 64 |\n| Train-only learner rollouts | 160 |\n| Candidate learner states | 1,276 |\n| Learner-state action-disagreement candidates | 387 |\n| Selected learner-state rows | 400 |\n| Selected rows with an actual disagreement | **175** |\n| Shared expert anchors per arm | 400 |\n| Total rows per training arm | **800** |\n| Identical realized prompt-completion pairs across arms | 452 |\n\nThe **452** realized identical pairs include the 400 intentionally shared anchors and 52 additional coincidental identical examples. Hence **do not claim all 400 differentiated corrective rows are errors**. The two published final training files are each 800 unique prompt-completion rows. Neither file contains direct development or sealed task IDs, although this simple leakage check does not establish statistical independence across a shared task generator.\n\n### Original SW12 file commitments\n\n- Train tasks SHA-256: `a41783c2370f2a9a33c90464c055b14161da028d11c273991b07e2f14c350793`\n- Development tasks SHA-256: `5fce6aa19f454549d5379d2858bff47c0e65ac04f28d523825fbc2b8437adfce`\n- Sealed tasks SHA-256: `3b91477db786aac7dd2f58ff0bc9e28f8033766e06d1d281625a7228731fba88`\n- Corrective v3 training SHA-256: `50adf52ca783358f3a9504ea65e17599e8181f374fb18a0e13153c4a40acfd50`\n- Rehearsal v3 training SHA-256: `cdf338acf469d8c7ffeb4e9c2749d1bb932d79d0a76fb7b3d9fe61740763671a`\n\nThe archived SW12 `selection_audit.json`, `precommit.json`, `final_provenance.json`, source notebook and per-episode JSONs provide the underlying audit trail.\n\n## SW17 continuation (source design, not a substitute for saved results)\n\nSW17 has **10** failure families, introducing approval-not-required and lease-expired cases. The source-defined design uses 240 training, 40 development and 80 sealed tasks, plus 160 train-only learner rollouts. It expands heldout lexical/service/region and configuration-field variation; both v4 arms again use 800 rows and 400 shared anchors, start from the same SW09 adapter, and train for 80 steps. An initial four-read bootstrap is held identical across policies, with no additional deterministic repair controller. The finalized source `selection_audit.json`, saved training rows, precommit and evaluation episodes should be published *without replacing them with these design targets*. SW17 still uses the same family of synthetic workflow environments; **it is not an independent external benchmark**.\n\n## Dataset fields and reproducibility\n\n- `*_training.jsonl`: each row contains `prompt` (observed history + action grammar) and `completion` (exact one-action JSON array). Corrective labels are tool actions, not free-form human rationales.\n- `*_tasks.jsonl`: scenario specifications, including goal, authorized state, failure-family ID, check-count and task split. Task specification contains hidden simulator metadata and must **not** be passed wholesale to the agent during evaluation.\n- `collection/*.json`: saved train-only model actions, tool observations, run metadata and scores.\n- `selection_audit.json`: coverage and label selection, including count of actual corrections.\n- `precommit.json`: seed and task-split checksums established before model evaluation.\n- `evaluation/**/*.json` and scorecards: retained strict outcomes and execution evidence.\n\nRecreate from the archived source notebooks in `notebooks/` using the exact base and starting adapters, pinned libraries, fixed generator seeds and task-identity commitments. Preserve the train/dev/sealed separation. **Do not include sealed test tasks or test traces in training.**\n\n## Methodological boundaries\n\nThe simulator and reference policy were developed together. Reference-policy labels are procedural ground truth *within the simulator*, not independent human expert annotations. Some failures are deterministic scripted injections. The 64-case SW12 sealed test is small and shares grammar with training. In SW12, matched rehearsal slightly exceeds corrective training on strict success (**47/64 vs 46/64**), so the data do **not** demonstrate that learner-state corrections uniquely outperform ordinary rehearsal. Strong runtime-controller gains must be reported separately.\n\nThis documentation follows the questions recommended by [Datasheets for Datasets](https://arxiv.org/abs/1803.09010) and the [Hugging Face dataset card guide](https://huggingface.co/docs/hub/en/datasets-cards), including source, generation, potential bias, recommended usage, and limitations. These references guide **documentation**, not claims of external dataset certification.\n")
# Deterministic manifest for all original released dataset files.
release_hashes = {
    path.relative_to(records).as_posix(): hashlib.sha256(path.read_bytes()).hexdigest()
    for path in sorted(records.rglob("*"))
    if path.is_file() and path.name != "sha256_release_manifest.json"
}
(records / "sha256_release_manifest.json").write_text(
    json.dumps(release_hashes, indent=2) + "\n"
)


api.upload_folder(folder_path=str(records), repo_id=DATASET_REPO_ID, repo_type="dataset", commit_message="Publish verified SW12–SW16 training and evaluation evidence")
public_dataset = api.dataset_info(DATASET_REPO_ID)
public_files = {item.rfilename for item in public_dataset.siblings}
assert not public_dataset.private and "sw12/train_tasks.jsonl" in public_files
assert "sw12/grouped_summary.csv" in public_files
assert "sw12/collection/SW12-TRAIN-00-000.json" in public_files
assert "sw17/learner_corrective_v4_training.jsonl" in public_files
assert "sw17/rehearsal_v4_training.jsonl" in public_files
assert "sw17/paired_tests.csv" in public_files
assert "DATASET_METHOD.md" in public_files and "README.md" in public_files
print("Dataset publication verified:", f"https://huggingface.co/datasets/{DATASET_REPO_ID}")


## 4. Add published repositories to the required existing public collections

In [ ]:
for repo_id in (CORRECTIVE_MODEL_ID, REHEARSAL_MODEL_ID, CORRECTIVE_V4_ID, REHEARSAL_V4_ID):
    api.add_collection_item(collection_slug=MODEL_COLLECTION_SLUG, item_id=repo_id, item_type="model", exists_ok=True)

api.add_collection_item(collection_slug=DATA_COLLECTION_SLUG, item_id=DATASET_REPO_ID, item_type="dataset", exists_ok=True)

for slug in (MODEL_COLLECTION_SLUG, DATA_COLLECTION_SLUG):
    collection = api.get_collection(slug)
    print("Collection verified:", f"https://huggingface.co/collections/{slug}")
    print("Items:", [(item.item_type, item.item_id) for item in collection.items])

print("SUCCESS: Models/datasets published, verified, and added to collections.")
print("GitHub: https://github.com/githubshaurya/ShadowWork-SHIFT")


## 5. Independent anonymous verification of final public release


In [ ]:
# Final release gate: verify files without any Hugging Face authentication.
# This checks what judges can actually view, rather than only what the uploader can.
public_api = HfApi(token=False)
public_models = (
    CORRECTIVE_MODEL_ID, REHEARSAL_MODEL_ID, CORRECTIVE_V4_ID, REHEARSAL_V4_ID,
)
for repo_id in public_models:
    info = public_api.model_info(repo_id, files_metadata=True, token=False)
    names = {entry.rfilename: entry for entry in info.siblings}
    assert not info.private and not info.gated, ("Not anonymously accessible", repo_id)
    assert {"adapter_model.safetensors", "adapter_config.json", "README.md"} <= set(names), repo_id
    assert (names["adapter_model.safetensors"].size or 0) > 100_000_000, (
        "LoRA weight upload appears incomplete", repo_id
    )
    print("ANONYMOUS PUBLIC MODEL VERIFIED:", repo_id, flush=True)

data_info = public_api.dataset_info(DATASET_REPO_ID, token=False)
all_public_files = {entry.rfilename for entry in data_info.siblings}
assert not data_info.private
required_dataset_paths = {
    "README.md", "DATASET_METHOD.md", "sha256_release_manifest.json",
    "sw12/train_tasks.jsonl", "sw12/dev_tasks.jsonl", "sw12/sealed_tasks.jsonl",
    "sw12/selection_audit.json", "sw12/learner_corrective_v3_training.jsonl",
    "sw12/rehearsal_v3_training.jsonl", "sw12/collection/SW12-TRAIN-00-000.json",
    "sw17/train_tasks.jsonl", "sw17/dev_tasks.jsonl", "sw17/sealed_tasks.jsonl",
    "sw17/selection_audit.json", "sw17/learner_corrective_v4_training.jsonl",
    "sw17/rehearsal_v4_training.jsonl", "sw17/policy_summary.csv",
    "sw17/paired_tests.csv", "sw17/learning_gate.json",
}
missing = required_dataset_paths - all_public_files
assert not missing, ("Missing public dataset files", sorted(missing))
print("ANONYMOUS PUBLIC DATASET VERIFIED:", DATASET_REPO_ID, "files:", len(all_public_files))

model_collection = public_api.get_collection(MODEL_COLLECTION_SLUG, token=False)
dataset_collection = public_api.get_collection(DATA_COLLECTION_SLUG, token=False)
published_model_items = {
    item.item_id for item in model_collection.items if item.item_type == "model"
}
published_dataset_items = {
    item.item_id for item in dataset_collection.items if item.item_type == "dataset"
}
assert set(public_models) <= published_model_items, (
    "One or more models missing from public HF collection", published_model_items
)
assert DATASET_REPO_ID in published_dataset_items, (
    "Published dataset missing from public HF dataset collection"
)
print("PUBLIC RELEASE GATE PASS: four model adapters, exact training data,")
print("two collections, episode evidence, and dataset method visible without login.")
print("PUBLIC MODEL COLLECTION:", f"https://huggingface.co/collections/{MODEL_COLLECTION_SLUG}")
print("PUBLIC DATA COLLECTION:", f"https://huggingface.co/collections/{DATA_COLLECTION_SLUG}")
